# Module 9: RAG & LLM Applications
## Vector Databases: 02 - ChromaDB Architecture & Persistence

### Theoretical Overview
**ChromaDB** is an open-source, developer-friendly embedding database designed specifically for LLM applications. Key capabilities:
- Embedded in-memory or persistent SQLite/DuckDB backing store.
- Built-in metadata filtering (`where={"category": "finance"}`).
- Built-in support for embedding functions or pre-computed embeddings.

In [ ]:
# ChromaDB Simulation: Emulating collection operations
class MockChromaCollection:
    def __init__(self, name: str):
        self.name = name
        self.store = {}
        
    def add(self, ids, documents, metadatas, embeddings):
        for i, doc, meta, emb in zip(ids, documents, metadatas, embeddings):
            self.store[i] = {"document": doc, "metadata": meta, "embedding": np.array(emb)}
            
    def query(self, query_embeddings, n_results=2, where=None):
        q = np.array(query_embeddings[0])
        matches = []
        for doc_id, data in self.store.items():
            if where:
                skip = False
                for k, v in where.items():
                    if data["metadata"].get(k) != v:
                        skip = True
                        break
                if skip: continue
            # Cosine similarity
            u = data["embedding"]
            score = float(np.dot(u, q) / (np.linalg.norm(u) * np.linalg.norm(q) + 1e-9))
            matches.append((doc_id, score, data["document"], data["metadata"]))
            
        matches.sort(key=lambda x: x[1], reverse=True)
        top = matches[:n_results]
        return {
            "ids": [[m[0] for m in top]],
            "documents": [[m[2] for m in top]],
            "metadatas": [[m[3] for m in top]],
            "distances": [[1.0 - m[1] for m in top]]
        }

coll = MockChromaCollection("ai_knowledge_base")
coll.add(
    ids=["doc1", "doc2", "doc3"],
    documents=["Chroma uses HNSW for nearest neighbors.", "PostgreSQL pgvector stores embeddings in SQL.", "PyTorch is a deep learning framework."],
    metadatas=[{"topic": "vectordb"}, {"topic": "vectordb"}, {"topic": "framework"}],
    embeddings=[[0.9, 0.1, 0.2], [0.85, 0.15, 0.25], [0.1, 0.9, 0.8]]
)

res = coll.query(query_embeddings=[[0.88, 0.12, 0.22]], n_results=2, where={"topic": "vectordb"})
print("ChromaDB Query Results:")
for did, doc in zip(res["ids"][0], res["documents"][0]):
    print(f"- [{did}]: {doc}")